In [ ]:
from pathlib import Path
import os
import shutil

import pandas as pd
from IPython.display import display


In [20]:
INPUT_DIR = "pyramid-forcing-60s/"
OUTPUT_DIR = "data/26Mar25-PyramidForcingView/pyramid-forcing-60s"
DOWNLOAD_ALL_VIDEOS = False
TARGET_VIDEO_NAME = "video_127.mp4"
USE_HF_MIRROR = True

In [21]:
VIDEO_SUFFIXES = (".mp4", ".mov", ".avi", ".mkv", ".webm")
HF_MIRROR_ENDPOINT = "https://hf-mirror.com"
HF_OFFICIAL_ENDPOINT = "https://huggingface.co"

# huggingface_hub 1.8.0 reads acceleration flags at import time.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"

from huggingface_hub import HfApi, hf_hub_download

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")

def resolve_output_dir(output_dir: str) -> Path:
    raw = Path(output_dir)
    return raw if raw.is_absolute() else resolve_repo_root() / raw

def normalize_input_dir(input_dir: str) -> str:
    value = input_dir.strip()
    if not value:
        raise ValueError("INPUT_DIR must not be empty.")
    return value if value.endswith("/") else f"{value}/"

INPUT_PREFIX = normalize_input_dir(INPUT_DIR)
HF_ENDPOINT = os.environ.get(
    "HF_ENDPOINT",
    HF_MIRROR_ENDPOINT if USE_HF_MIRROR else HF_OFFICIAL_ENDPOINT,
)
REPO_ID = "kv-compression/Pyramid-Forcing-Experiments"
OUTPUT_PATH = resolve_output_dir(OUTPUT_DIR)
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
{
    "input_dir": INPUT_PREFIX,
    "output_dir": str(OUTPUT_PATH),
    "download_all_videos": DOWNLOAD_ALL_VIDEOS,
    "target_video_name": TARGET_VIDEO_NAME,
    "use_hf_mirror": USE_HF_MIRROR,
    "hf_endpoint": HF_ENDPOINT,
    "hf_hub_enable_hf_transfer": os.environ["HF_HUB_ENABLE_HF_TRANSFER"],
    "hf_xet_high_performance": os.environ["HF_XET_HIGH_PERFORMANCE"],
}

{'input_dir': 'pyramid-forcing-60s/',
 'output_dir': '/home/winbeau/Tools/jupyter-plot/data/26Mar25-PyramidForcingView/pyramid-forcing-60s',
 'download_all_videos': False,
 'target_video_name': 'video_127.mp4',
 'use_hf_mirror': True,
 'hf_endpoint': 'https://hf-mirror.com',
 'hf_hub_enable_hf_transfer': '1',
 'hf_xet_high_performance': '1'}

In [ ]:
if DOWNLOAD_ALL_VIDEOS:
    api = HfApi(endpoint=HF_ENDPOINT)
    files = api.list_repo_files(repo_id=REPO_ID, repo_type="dataset")
    target_files = sorted(
        path
        for path in files
        if path.startswith(INPUT_PREFIX) and Path(path).suffix.lower() in VIDEO_SUFFIXES
    )
    if not target_files:
        raise FileNotFoundError(f"No video files matched dataset group: {INPUT_PREFIX}")
else:
    target_files = [f"{INPUT_PREFIX}{TARGET_VIDEO_NAME}"]

print(f"will download {len(target_files)} files")

rows = []
for repo_path in target_files:
    destination_path = OUTPUT_PATH / Path(repo_path).name
    destination_path.parent.mkdir(parents=True, exist_ok=True)

    if destination_path.exists():
        print("hit local file:", destination_path)
        rows.append({
            "repo_path": repo_path,
            "local_path": str(destination_path),
            "status": "local-hit",
        })
        continue

    print("downloading:", repo_path)
    cached_path = Path(
        hf_hub_download(
            repo_id=REPO_ID,
            repo_type="dataset",
            filename=repo_path,
            endpoint=HF_ENDPOINT,
        )
    )

    if destination_path.exists():
        destination_path.unlink()

    try:
        os.link(cached_path, destination_path)
        link_mode = "hardlink"
    except OSError:
        try:
            destination_path.symlink_to(cached_path)
            link_mode = "symlink"
        except OSError:
            shutil.copy2(cached_path, destination_path)
            link_mode = "copy"

    rows.append({
        "repo_path": repo_path,
        "local_path": str(destination_path),
        "status": f"downloaded-{link_mode}",
    })

result_df = pd.DataFrame(rows)
display(result_df)
print("done")
